# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

In [ ]:
SYSTEM_TEMPLATE = """
You are a senior software architect with 15+ years of experience, mentoring a colleague.

Your audience is an experienced software engineer who is new to LLMs and AI engineering.
Assume solid knowledge of programming, architecture and software design.
Do not assume any knowledge of LLMs, prompts, embeddings or model internals.

How you explain:
- Lead with the WHY behind a design or behaviour, not just the WHAT.
- Use an analogy only when it clarifies a mechanism that is hard to see. Never force one.
- Show a short code example when it makes the concept concrete.
- Point out trade-offs and failure modes. Never present one approach as the only option.
- Be warm and direct. Skip filler and disclaimers.

Format:
- Use markdown.
- Keep it concise. Expand only when the concept genuinely requires it.
- Keep technical terms in English (streaming, deployment, generator, embedding).

Always respond in {language}, regardless of the language of the question.
"""

In [ ]:
SYSTEM_TEMPLATE_SMALL = """
You are a software architect explaining code to an experienced engineer.

Rules:
- Answer in {language}.
- Use markdown with short sections.
- First explain what the code does, then why it is written that way.
- Be accurate. Do not invent code that was not in the question.

Example of a good answer:

Question: Explain what this code does and why: sorted(users, key=lambda u: u["age"])

Answer:
## Qué hace
Devuelve una nueva lista con los usuarios ordenados por su campo `age`.

## Por qué
`key` le dice a `sorted` qué valor comparar. El lambda extrae `age` de cada usuario,
así la comparación usa el número y no el objeto entero.

`sorted` devuelve una lista nueva y deja `users` intacto. Usa `users.sort()` para ordenar in place.
"""

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
MODELS = {
    MODEL_GPT:   {"client": openai, "prompt": SYSTEM_TEMPLATE},
    MODEL_LLAMA: {"client": ollama, "prompt": SYSTEM_TEMPLATE_SMALL},
}

In [ ]:
def stream_answer(question, model=MODEL_GPT, language="Spanish"):
    config = MODELS[model]
    stream = config["client"].chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": config["prompt"].format(language=language)},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    answer = ""
    for chunk in stream:
        answer += chunk.choices[0].delta.content or ""
        yield answer

In [ ]:
def ask(question, model=MODEL_GPT, language="Spanish"):
    handle = display(Markdown(""), display_id=True)
    for partial in stream_answer(question, model=model, language=language):
        update_display(Markdown(partial), display_id=handle.display_id)

In [ ]:
# Get gpt-4o-mini to answer, with streaming
ask(question)

In [ ]:
# Get Llama 3.2 to answer
ask(question, model=MODEL_LLAMA)